In [ ]:
# ngrok config add-authtoken 

In [ ]:
# Runtime > Change Runtime Type > A100 GPU

model_url = "https://huggingface.co/hugging-quants/Llama-3.2-1B-Instruct-Q4_K_M-GGUF/resolve/main/llama-3.2-1b-instruct-q4_k_m.gguf"
model_path = "models/llama-3.2-1b-instruct-q4_k_m.gguf"
ngrok_authtoken = "YOUR_TOKEN_HERE"

In [3]:
import os
import time
import subprocess
import requests

port = 8000
n_ctx = 2048

!apt-get update -qq
!apt-get install -y -qq curl wget
!pip -q install --upgrade pip
!pip -q install "llama-cpp-python[server]" pyngrok requests

os.makedirs("models", exist_ok=True)

if not os.path.exists(model_path):
    print("Downloading model...")
    r = requests.get(model_url, stream=True, timeout=60)
    r.raise_for_status()
    with open(model_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
    print("Model downloaded:", model_path)
else:
    print("Model already exists:", model_path)



W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Model already exists: models/llama-3.2-1b-instruct-q4_k_m.gguf


In [4]:
# this starts llama.cpp server
server_cmd = [
    "python", "-m", "llama_cpp.server",
    "--model", model_path,
    "--host", "0.0.0.0",
    "--port", str(port),
    "--n_ctx", str(n_ctx),
    "--n_gpu_layers", "-1",
]

server_proc = subprocess.Popen(server_cmd)
time.sleep(10)

if server_proc.poll() is not None:
    raise RuntimeError("llama-cpp server failed to start.")

print(f"llama-cpp server running on http://127.0.0.1:{port}")

# this starts ngrok tunnel
from pyngrok import ngrok

if ngrok_authtoken:
    ngrok.set_auth_token(ngrok_authtoken)
else:
    raise ValueError("Set ngrok_authtoken before running this notebook.")

public_url = ngrok.connect(port, "http").public_url
print("Public URL:", public_url)



llama-cpp server running on http://127.0.0.1:8000
Public URL: https://ba22-34-124-208-84.ngrok-free.app


In [ ]:
# Run this code locally to make a request to the llama-cpp server:

import requests

system_prompt = """
TODO: Write a system prompt to instruct the server to translate the input sequence
"""

def translate_gloss(gloss_text, temperature=0.2, max_tokens=128):
    url = f"{public_url}/v1/chat/completions"
    payload = {
        "model": "local-model",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": gloss_text}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    resp = requests.post(url, json=payload, timeout=120)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"].strip()

example = "I GO STORE"
print("Input:", example)
print("Output:", translate_gloss(example))



Input: PRO.1 GO STORE FINISH
Output: The store is now finished.
